In [3]:
# Defining an enviroment for the agent (logistics center)
import numpy as np
import random as rd


class logistics_center:

    def __init__(self, num_docks = 3, num_SKU = 3):
        self.num_docks = num_docks
        self.num_SKU = num_SKU

    def create_warehouse(self):

        warehouse = np.zeros((self.num_docks, (self.num_docks + 2)))

        for i in range(self.num_docks):
            warehouse[i,0] = 1

        j = 0 

        while j < self.num_SKU:
            a , b = rd.randint(0, (self.num_docks -1)), rd.randint(1, (self.num_docks + 1))
            if warehouse[a,b] == 0:
                warehouse[a, b] = 2
                j += 1

        return warehouse

    def reset(self):
        self.warehouse = self.create_warehouse()
        self.state = State(self.warehouse)          
        self.state.create_state(self.warehouse)
        self.agent = Agent(self.warehouse, self.state)

    def episode(self):
        self.reset()
        
        while True:

            action = rd.choice([
            self.agent.action_up,
            self.agent.action_down,
            self.agent.action_left,
            self.agent.action_right
        ])
            action()

            reward = self.reward(self.agent)

            print("Position:", self.agent.position)
            print("Reward:", reward)


            if self.state.has_SKU[1] >= 1 and self.state.deadline[1] >= 0:
                print("Success!")
                self.reset()
                break

            if self.state.deadline[0] >= 10:
                print("Fail")
                self.reset()
                break

    def reward(self, agent):
        reward = 0

        reward -= 1

        if agent.has_SKU[0] == 1 and agent.just_picked_SKU:
            reward +=5

        if agent.has_SKU[1] >= 1 and agent.just_delivered_SKU:
            reward += 10

        if agent.deadline[1] == 0 and agent.has_SKU[1] == 0:
            reward -= 10 

        return reward


In [4]:
#defining the state, which is going to have the SKU requisition, the agent position, where does he need to go, deadline and if he is carrying an SKU

class State(logistics_center):

    def __init__(self, created_warehouse):
        self.created_warehouse = created_warehouse

    def create_state(self, created_warehouse):

        state_list = []
        
        empty_positions = np.argwhere(created_warehouse == 0)
        self.position = np.array(rd.choice(empty_positions))
        self.position = [int(x)for x in self.position]
        state_list.append(self.position)
        

        dock_positions = np.argwhere(created_warehouse == 1)
        self.delivery_dock = np.array(rd.choice(dock_positions))
        self.delivery_dock = [int(x)for x in self.delivery_dock]
        state_list.append(self.delivery_dock)
        

        sku_positions = np.argwhere(created_warehouse == 2)
        self.SKU_requisition = np.array(rd.choice(sku_positions))
        self.SKU_requisition = [int(x)for x in self.SKU_requisition]
        state_list.append(self.SKU_requisition)



        self.deadline = np.array([0, 10]) # the agent can do 10 "moves" to get to accomplish his mission, the first item 0 = no carrying, 1 = carrying SKU
        state_list.append(self.deadline)

        self.has_SKU = np.array([0, 0]) # [0] = 0 , no SKU, [1] = 1, has SKU; [1] how many SKUs were delivered
        state_list.append(self.has_SKU)
    
        state = np.array(state_list)

        return state
        

In [5]:
# defining the agent, which is going to have actions and get the state and warehouse created
# the agent can move upwards, downwards and go to the left or to the right, movimentation implemented via matrix 
# we also change the state here, by moving the agent, the dealine is changed as well as the has_SKU state

class Agent():

    def __init__(self, warehouse_created, state):

        self.warehouse_created = warehouse_created
        self.state = state  
        self.just_picked_SKU = False
        self.just_delivered_SKU = False

    @property
    def position(self):
        return self.state.position

    @property
    def deadline(self):
        return self.state.deadline

    @property
    def has_SKU(self):
        return self.state.has_SKU

    def action_up(self):
        self.just_picked_SKU = False
        self.just_delivered_SKU = False
        self.matrix_up = np.array([-1, 0])

        if self.position[0] - 1 >= 0:
            new_position = np.array(self.position) + self.matrix_up
            self.state.position[:] = [int(x) for x in new_position]   
            self.state.deadline += np.array([1, -1])
            if self.warehouse_created[self.position[0], self.position[1]] == 2 and self.has_SKU[0] == 0:
                self.state.has_SKU += np.array([1, 0])
                self.just_picked_SKU = True

            if self.warehouse_created[self.position[0], self.position[1]] == 1 and self.has_SKU[0] == 1:
                self.state.has_SKU += np.array([-1, 1])
                self.just_delivered_SKU = True

    def action_down(self):
        self.just_picked_SKU = False
        self.just_delivered_SKU = False
        self.matrix_down = np.array([1, 0])

        if self.position[0] + 1 < len(self.warehouse_created):
            new_position = np.array(self.position) + self.matrix_down
            self.state.position[:] = [int(x) for x in new_position]
            self.state.deadline += np.array([1, -1])
            if self.warehouse_created[self.position[0], self.position[1]] == 2 and self.has_SKU[0] == 0:
                self.state.has_SKU += np.array([1, 0])
                self.just_picked_SKU = True

            if self.warehouse_created[self.position[0], self.position[1]] == 1 and self.has_SKU[0] == 1:
                self.state.has_SKU += np.array([-1, 1])
                self.just_delivered_SKU = True

    def action_left(self):
        self.just_picked_SKU = False
        self.just_delivered_SKU = False
        self.matrix_left = np.array([0, -1])

        if self.position[1] - 1 >= 0:
            new_position = np.array(self.position) + self.matrix_left
            self.state.position[:] = [int(x) for x in new_position]
            self.state.deadline += np.array([1, -1])
            if self.warehouse_created[self.position[0], self.position[1]] == 2 and self.has_SKU[0] == 0:
                self.state.has_SKU += np.array([1, 0])
                self.just_picked_SKU = True

            if self.warehouse_created[self.position[0], self.position[1]] == 1 and self.has_SKU[0] == 1:
                self.state.has_SKU += np.array([-1, 1])
                self.just_delivered_SKU = True

    def action_right(self):
        self.just_picked_SKU = False
        self.just_delivered_SKU = False
        self.matrix_right = np.array([0, 1])

        if self.position[1] + 1 < len(self.warehouse_created[0]):
            new_position = np.array(self.position) + self.matrix_right
            self.state.position[:] = [int(x) for x in new_position]
            self.state.deadline += np.array([1, -1])
            if self.warehouse_created[self.position[0], self.position[1]] == 2 and self.has_SKU[0] == 0:
                self.state.has_SKU += np.array([1, 0])
                self.just_picked_SKU = True

            if self.warehouse_created[self.position[0], self.position[1]] == 1 and self.has_SKU[0] == 1:
                self.state.has_SKU += np.array([-1, 1])
                self.just_delivered_SKU = True